In [1]:
from __future__ import annotations

import json
from pathlib import Path
from statistics import mean

import pandas as pd


# ---------- Paths ----------
def find_repo_root(start: Path) -> Path:
    """Walk upward until we find the repository root."""
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "tasks" / "tasks_100.json").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")


ROOT = find_repo_root(Path.cwd().resolve())
RESULTS_DIR = ROOT / "results"
TASKS_PATH = ROOT / "src" / "tasks" / "tasks_100.json"

RUN_ROOTS = {
    "No CD": RESULTS_DIR / "bt_nocd_all100_parallel",
    "CD": RESULTS_DIR / "bt_cd_all100_parallel",
}


# ---------- Benchmark task mapping ----------
TASK_TYPE_ORDER = [
    ("face_target", "Face(Easy)"),
    ("go_to_target", "GoTo(Easy)"),
    ("go_around_obstacle", "Around(Med)"),
    ("move_to_closest_target", "Closest(Hard)"),
    ("go_to_multiple_targets", "MultiGoTo(Hard)"),
]


def load_tasks_index(tasks_path: Path) -> tuple[dict[str, str], dict[str, int]]:
    tasks = json.loads(tasks_path.read_text(encoding="utf-8"))
    task_id_to_type: dict[str, str] = {}
    expected_per_type = {task_type: 0 for task_type, _ in TASK_TYPE_ORDER}

    for task in tasks:
        task_id = str(task["task_id"])
        task_type = task["task_type"]
        task_id_to_type[task_id] = task_type
        if task_type in expected_per_type:
            expected_per_type[task_type] += 1

    return task_id_to_type, expected_per_type


def clean_model_name(model_id: str) -> str:
    """Normalize names for display by removing namespace."""
    return model_id.split("/", 1)[1] if "/" in model_id else model_id


def collect_available_models() -> set[str]:
    """Collect model directory names present in either CD or No-CD roots."""
    model_dirs: set[str] = set()
    for run_root in RUN_ROOTS.values():
        if not run_root.exists():
            continue
        for child in run_root.iterdir():
            if child.is_dir():
                model_dirs.add(child.name)
    return model_dirs


def load_mode_metrics(
    run_root: Path,
    model_dir: str,
    task_id_to_type: dict[str, str],
    expected_per_type: dict[str, int],
) -> dict[str, tuple[float | None, float | None]] | None:
    """Return per-task-type (SA, TA) for one model+mode."""
    model_root = run_root / model_dir
    main_results_path = model_root / "main_results.json"
    tasks_dir = model_root / "tasks"

    if not main_results_path.exists():
        return None

    main_results = json.loads(main_results_path.read_text(encoding="utf-8"))

    metrics: dict[str, tuple[float | None, float | None]] = {
        task_type: (None, None) for task_type, _ in TASK_TYPE_ORDER
    }

    # Fallback: TA from main_results when task files are missing. SA is unavailable.
    if not tasks_dir.exists():
        completion_map = main_results.get("completion_pct_by_task_type", {})
        for task_type, _ in TASK_TYPE_ORDER:
            ta = completion_map.get(f"{task_type}_pct")
            metrics[task_type] = (None, float(ta) if ta is not None else None)
        return metrics

    sums = {
        task_type: {
            "total_bt": 0,
            "valid_structure": 0,
            "completed": 0,
            "seen": 0,
        }
        for task_type, _ in TASK_TYPE_ORDER
    }

    for task_file in tasks_dir.glob("task_*.json"):
        task_id = task_file.stem.removeprefix("task_")
        task_type = task_id_to_type.get(task_id)
        if task_type not in sums:
            continue

        task_result = json.loads(task_file.read_text(encoding="utf-8"))
        sums[task_type]["seen"] += 1
        sums[task_type]["total_bt"] += int(task_result.get("bt_count", 0))
        sums[task_type]["valid_structure"] += int(task_result.get("valid_structure_count", 0))
        sums[task_type]["completed"] += 1 if bool(task_result.get("task_completion", False)) else 0

    for task_type, _ in TASK_TYPE_ORDER:
        expected = expected_per_type[task_type]
        seen = sums[task_type]["seen"]

        # Missing benchmark tasks for that type => unavailable
        if seen < expected or expected == 0:
            metrics[task_type] = (None, None)
            continue

        total_bt = sums[task_type]["total_bt"]
        valid_structure = sums[task_type]["valid_structure"]
        completed = sums[task_type]["completed"]

        sa = (valid_structure / total_bt) * 100 if total_bt > 0 else None
        ta = (completed / expected) * 100 if expected > 0 else None
        metrics[task_type] = (sa, ta)

    return metrics


def fmt(value: float | None) -> str:
    # Treat both None and NaN as unavailable entries.
    if value is None or pd.isna(value):
        return "X"
    return f"{float(value):.1f}"


# ---------- Build combined records ----------
task_id_to_type, expected_per_type = load_tasks_index(TASKS_PATH)
available_model_dirs = collect_available_models()

preferred_order = [
    "Qwen2.5-1.5B-Instruct",
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-14B-Instruct",
    "Qwen3-8B",
    "Qwen3-14B",
    "Llama-3.1-8B-Instruct",
    "Llama-3.2-3B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Ministral-8B-Instruct-2410",
    "phi-4",
    "granite-3.3-8b-instruct",
    "gemma-2-9b-it",
    "DeepSeek-R1-Distill-Qwen-14B",
]

# Map model_dir -> display name from any available main_results.
model_dir_to_name: dict[str, str] = {}
for run_root in RUN_ROOTS.values():
    if not run_root.exists():
        continue
    for model_dir in available_model_dirs:
        main_results_path = run_root / model_dir / "main_results.json"
        if not main_results_path.exists():
            continue
        main_results = json.loads(main_results_path.read_text(encoding="utf-8"))
        model_id = str(main_results.get("model_id", model_dir))
        model_dir_to_name[model_dir] = clean_model_name(model_id)

name_to_dir = {name: model_dir for model_dir, name in model_dir_to_name.items()}
ordered_names = preferred_order[:]  # Always include this fixed model set.

records = []
for model_name in ordered_names:
    model_dir = name_to_dir.get(model_name)
    for dec_label in ["No CD", "CD"]:
        mode_metrics = (
            load_mode_metrics(
                RUN_ROOTS[dec_label],
                model_dir,
                task_id_to_type,
                expected_per_type,
            )
            if model_dir is not None
            else None
        )

        row = {"Model": model_name, "Dec.": dec_label}
        sa_values: list[float] = []
        ta_values: list[float] = []

        for task_type, task_label in TASK_TYPE_ORDER:
            if mode_metrics is None:
                sa, ta = None, None
            else:
                sa, ta = mode_metrics[task_type]

            row[f"{task_label} SA"] = sa
            row[f"{task_label} TA"] = ta

            if sa is not None:
                sa_values.append(sa)
            if ta is not None:
                ta_values.append(ta)

        row["Avg. SA"] = mean(sa_values) if len(sa_values) == len(TASK_TYPE_ORDER) else None
        row["Avg. TA"] = mean(ta_values) if len(ta_values) == len(TASK_TYPE_ORDER) else None
        records.append(row)


df = pd.DataFrame(records)

# Render final grouped table directly in this cell.
from IPython.display import display

ordered_pairs = [
    ("Face(Easy)", "SA"), ("Face(Easy)", "TA"),
    ("GoTo(Easy)", "SA"), ("GoTo(Easy)", "TA"),
    ("Around(Med)", "SA"), ("Around(Med)", "TA"),
    ("Closest(Hard)", "SA"), ("Closest(Hard)", "TA"),
    ("MultiGoTo(Hard)", "SA"), ("MultiGoTo(Hard)", "TA"),
    ("Avg.", "SA"), ("Avg.", "TA"),
]
flat_to_grouped = {
    "Face(Easy) SA": ("Face(Easy)", "SA"), "Face(Easy) TA": ("Face(Easy)", "TA"),
    "GoTo(Easy) SA": ("GoTo(Easy)", "SA"), "GoTo(Easy) TA": ("GoTo(Easy)", "TA"),
    "Around(Med) SA": ("Around(Med)", "SA"), "Around(Med) TA": ("Around(Med)", "TA"),
    "Closest(Hard) SA": ("Closest(Hard)", "SA"), "Closest(Hard) TA": ("Closest(Hard)", "TA"),
    "MultiGoTo(Hard) SA": ("MultiGoTo(Hard)", "SA"), "MultiGoTo(Hard) TA": ("MultiGoTo(Hard)", "TA"),
    "Avg. SA": ("Avg.", "SA"), "Avg. TA": ("Avg.", "TA"),
}
actual_table = df.set_index(["Model", "Dec."]).rename(columns=flat_to_grouped)
actual_table = actual_table[[flat_to_grouped[k] for k in flat_to_grouped]]
actual_table.columns = pd.MultiIndex.from_tuples(actual_table.columns)
actual_table = actual_table.reindex(columns=pd.MultiIndex.from_tuples(ordered_pairs))
display(actual_table.apply(lambda col: col.map(fmt)))

# ---------- LaTeX table builder ----------
def build_latex_table(table_df: pd.DataFrame) -> str:
    lines: list[str] = []

    lines.append(r"\begin{tabular}{@{}l l cc cc cc cc cc cc@{}}")
    lines.append(r"\toprule")
    lines.append(
        r"& "
        + r"& \multicolumn{2}{c}{\textbf{Face}(Easy)}"
        + r"& \multicolumn{2}{c}{\textbf{GoTo}(Easy)}"
        + r"& \multicolumn{2}{c}{\textbf{Around}(Med)}"
        + r"& \multicolumn{2}{c}{\textbf{Closest}(Hard)}"
        + r"& \multicolumn{2}{c}{\textbf{MultiGoTo}(Hard)}"
        + r"& \multicolumn{2}{c}{\textbf{Avg.}} \\"
    )
    lines.append(
        r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}"
        r"\cmidrule(lr){9-10}\cmidrule(lr){11-12}\cmidrule(lr){13-14}"
    )
    lines.append(
        r"\textbf{Model} & \textbf{Dec.}"
        + r"& SA & TA & SA & TA & SA & TA & SA & TA & SA & TA & SA & TA \\"
    )
    lines.append(r"\midrule")

    grouped_models = list(table_df["Model"].drop_duplicates())
    for model_index, model_name in enumerate(grouped_models):
        model_rows = table_df[table_df["Model"] == model_name]

        row_no_cd = (
            model_rows[model_rows["Dec."] == "No CD"].iloc[0]
            if (model_rows["Dec."] == "No CD").any()
            else None
        )
        row_cd = (
            model_rows[model_rows["Dec."] == "CD"].iloc[0]
            if (model_rows["Dec."] == "CD").any()
            else None
        )

        def render_cells(row) -> list[str]:
            if row is None:
                return ["X"] * 12

            out: list[str] = []
            for _, label in TASK_TYPE_ORDER:
                out.append(fmt(row[f"{label} SA"]))
                out.append(fmt(row[f"{label} TA"]))
            out.append(fmt(row["Avg. SA"]))
            out.append(fmt(row["Avg. TA"]))
            return out

        no_cd_cells = " & ".join(render_cells(row_no_cd))
        cd_cells = " & ".join(render_cells(row_cd))

        lines.append(rf"\multirow{{2}}{{*}}{{{model_name}}}")
        lines.append(rf"& No CD & {no_cd_cells} \\")
        lines.append(rf"& CD    & {cd_cells} \\")

        if model_index < len(grouped_models) - 1:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)

Face(Easy)       GoTo(Easy)        \
                                           SA    TA         SA    TA   
Model                        Dec.                                      
Qwen2.5-1.5B-Instruct        No CD       53.4  10.0        5.0   0.0   
                             CD          50.0  10.0        5.0   0.0   
Qwen2.5-3B-Instruct          No CD      100.0  10.0       98.3   0.0   
                             CD         100.0  10.0      100.0   0.0   
Qwen2.5-7B-Instruct          No CD       96.3  15.0      100.0  10.0   
                             CD         100.0  15.0      100.0  10.0   
Qwen2.5-14B-Instruct         No CD       68.0  30.0      100.0  10.0   
                             CD          95.9  35.0      100.0  10.0   
Qwen3-8B                     No CD      100.0  10.0      100.0  10.0   
                             CD         100.0  10.0      100.0  10.0   
Qwen3-14B                    No CD      100.0  15.0      100.0   5.0   
                             CD         100.0  15.0      100.0   5.0   
Llama-3.1-8B-Instruct        No CD       33.3  10.0       33.3  10.0   
                             CD          96.4  10.0      100.0  15.0   
Llama-3.2-3B-Instruct        No CD        1.7   0.0        0.0   0.0   
                             CD          71.2  20.0       90.0   0.0   
Mistral-7B-Instruct-v0.3     No CD        0.0   0.0        0.0   0.0   
                             CD         100.0  10.0      100.0   0.0   
Ministral-8B-Instruct-2410   No CD       31.7  10.0        0.0   0.0   
                             CD          55.2  10.0        5.0   0.0   
phi-4                        No CD       70.6  25.0       90.7  15.0   
                             CD         100.0  30.0      100.0  15.0   
granite-3.3-8b-instruct      No CD      100.0  15.0       79.3   5.0   
                             CD         100.0  15.0       93.1   5.0   
gemma-2-9b-it                No CD      100.0  25.0      100.0  10.0   
                             CD         100.0  30.0      100.0  10.0   
DeepSeek-R1-Distill-Qwen-14B No CD        0.0   0.0        0.0   0.0   
                             CD         100.0  10.0      100.0  10.0   

                                   Around(Med)       Closest(Hard)        \
                                            SA    TA            SA    TA   
Model                        Dec.                                          
Qwen2.5-1.5B-Instruct        No CD        38.3   0.0          23.3   0.0   
                             CD           45.0   0.0          18.3   0.0   
Qwen2.5-3B-Instruct          No CD        48.3   5.0          74.1   5.0   
                             CD           61.7   5.0          87.9   5.0   
Qwen2.5-7B-Instruct          No CD        55.0   0.0         100.0   5.0   
                             CD          100.0   0.0         100.0   5.0   
Qwen2.5-14B-Instruct         No CD        60.3  10.0          96.1  30.0   
                             CD           96.5  15.0         100.0  30.0   
Qwen3-8B                     No CD       100.0   0.0         100.0   5.0   
                             CD          100.0   0.0         100.0   5.0   
Qwen3-14B                    No CD       100.0   0.0         100.0   5.0   
                             CD          100.0   0.0         100.0   5.0   
Llama-3.1-8B-Instruct        No CD        33.3   0.0          33.3  15.0   
                             CD          100.0   0.0          98.2  20.0   
Llama-3.2-3B-Instruct        No CD         0.0   0.0           0.0   0.0   
                             CD           98.3   0.0          91.7   0.0   
Mistral-7B-Instruct-v0.3     No CD         3.3   0.0           0.0   0.0   
                             CD           88.3   0.0         100.0   5.0   
Ministral-8B-Instruct-2410   No CD         0.0   0.0          33.3   5.0   
                             CD            0.0   0.0          63.8  10.0   
phi-4                        No CD        70.0   0.0  

In [2]:
# Intentionally left blank; table generation is done in Cell 0.
